**Notebook Feito por:** MSc. Eng. Paulo de Souza Silva  
**Data:** Julho de 2026  
**Conteúdo retirado e adaptado de livros e artigos sobre Galerkin Descontínuo**  
**Agradecimentos:** Um agradecimento ao Prof. Dr. Alberto Nogueira pela disponibilização dos scripts em Python para DG 1D

# **Aula 06 - Fundamentos Geométricos e as Matrizes Clássicas do DG 1D**

## **Introdução**

Até o momento no curso, construímos as ferramentas que servem de base matemática para o Método de Galerkin Descontínuo (DG), culminando na nossa Matriz de Diferenciação. A partir desta aula, começaremos a unir essas peças para montar a estrutura fundamental do nosso solver.

O nosso ponto de partida será a construção dos elementos geométricos e dos operadores espaciais que sustentarão todos os problemas 1D que pretendemos resolver. Neste encontro, você aprenderá a construir:

* A **Malha 1D**, responsável por discretizar o nosso domínio físico;
* O **Jacobiano**, essencial para o mapeamento entre os elementos reais e o domínio de referência;
* A nossa Base Oficial, consolidando o uso dos **Polinômios de Legendre** para a aproximação da solução. 
* A **Matriz de Massa**, que rege a inércia do sistema e possui propriedades que facilitam muito a sua inversão.
* A **Matriz de Rigidez**, que pode ser formulada de maneiras distintas a depender da física do problema (advecção pura ou com difusão) ou da estratégia numérica adotada.
* As **Lift Matrices** (Matrizes de Elevação), um conceito novo que será vital para o tratamento de fronteiras e para a construção do nosso operador principal **Lh**

> **Nota:** Para não poluir nosso notebook, vamos começar a utilizar um arquivo python denominado `solverDG1D` que contém todas as funções que desenvolvemos até aqui


## **Malha 1D**

Associado a todo problema que iremos resolver, precisamos definir uma malha unidimensional que representa o nosso domínio físico. 

Como estamos trabalhando em 1D, necessitamos apenas informar quantos elementos ($K$) pretendemos utilizar, ou seja, o controlador do nosso refino espacial $h$ da estratégia *hp-spectral*, e os limites espaciais do domínio real, dados por $x_{min}$ e $x_{max}$. Em todos os nossos casos, adotaremos para a criação da malha elementos equidistantes (de mesmo tamanho). 

Bebendo da fonte do Método dos Elementos Finitos (MEF), é padrão extrairmos três informações fundamentais da malha geométrica:
* O número total de vértices/nós da malha ($N_v$), que serve de parâmetro para as demais rotinas.
* As coordenadas reais dos vértices ($VX$), que nos ajudarão no mapeamento do domínio de referência para o domínio físico.
* A matriz de conectividade dos elementos ($EToV$), uma matriz de dimensão $K \times 2$ que nos informa quais vértices globais delimitam cada elemento. 

> **Nota:** No nosso caso unidimensional, é trivial saber que os elementos estão conectados em ordem crescente. No entanto, criar e carregar a matriz $EToV$ é uma prática de programação científica indispensável, pois em geometrias bidimensionais (2D) e tridimensionais (3D), ou em malhas não estruturadas, esse mapeamento de conectividade é o "coração" da topologia do *solver*.

*(Figura: Adicionar ilustração contendo a representação geométrica dos nós globais, os elementos $K$ e a indicação de mapeamento local-global).*

In [ ]:
import numpy as np
from mathDG1D import JacobiP, jacobi_gauss_quad, DMatrix1D


In [2]:
def MeshGen1D(K: int, xmin: float = 0.0, xmax: float = 1.0): 
    """ 
    Gerador de uma malha simples equidistante com K elementos.

    ### Parâmetros:
        K (int): Número de elementos da malha.
        xmin (float): Limite inferior do domínio real.
        xmax (float): Limite superior do domínio real.

    ### Retorna:
        Nv (int): Número total de vértices (nós).
        VX (array): Vetor de coordenadas globais dos vértices.
        EToV (array): Matriz de conectividade (K x 2) contendo os índices 
                      dos vértices que compõem cada elemento k.
    """
    Nv = K + 1  # Número total de vértices da malha
    VX = np.linspace(xmin, xmax, Nv) # Criação do vetor de coordenadas

    # Matriz de conectividade (Element to Vertex)
    EToV = np.zeros((K, 2), dtype=int) 
    for k in range(K): 
        EToV[k, 0] = k 
        EToV[k, 1] = k + 1

    return Nv, VX, EToV

In [34]:
K = 4 # numero de elementos para a malha
Nv, VX, EToV = MeshGen1D(K,0,2)
print(f"Numero de Nos: {Nv}\nCoordenadas da Malha: {VX}\nMapeamento dos Indices: \n{EToV}")


Numero de Nos: 5
Coordenadas da Malha: [0.  0.5 1.  1.5 2. ]
Mapeamento dos Indices: 
[[0 1]
 [1 2]
 [2 3]
 [3 4]]


---

## **Jacobiano**

Como vimos na aula 00 e também ao que aprendemos sobre as quadraturas, é necessário que a gente tenha a capacidade de mapear  
as coordenadas globais $x\in[x_{e}^{-},x_{e}^{+}]$ para um domínio local restrito da forma $\xi\in[-1,1]$ aplicando  transformações lineares no espaço 1D e também sermos capazes de sair do domínio local para as coordenadas globais.

<div align="center">
  <img src="https://raw.githubusercontent.com/properallan/CFD4SciML/main/DG/Lessons/images/aula00_jacobiano.png" alt="Mapeamento de coordenadas" width="800"><br>
  <em>Mapeamento de coordenadas</em>
</div>

Na aula 00, ainda definimos que se o tamanho do elemento for definido por $h_e = x_e^+ - x_e^-$ temos que

* Da coordenada global para o elemento padrão:
$$\xi_{e}(x)=2\frac{x-x_{e}^{-}}{h_e}-1$$

* Do elemento padrão isolado de volta à dimensão real:
$$x_{e}(\xi)=x_e^- + \frac{1+\xi}{2}h_e$$

No entanto, essa não é a única forma de escrever essas transformações. Veja que de forma similar, mas usando o tamanho do elemento de forma explicita, podemos reescrever a mudança da coordenada global para o elemento padrão:
$$\xi_{e}(x)=\frac{2}{x_e^+ - x_e^-} x - \frac{x_e^+ + x_e^-}{x_e^+ - x_e^-}$$

elemento padrão isolado de volta à dimensão real como:
$$x_{e}(\xi) = \frac{x_e^+ - x_e^- }{2}\xi + \frac{x_e^- + x_e^+}{2}$$


Note que se quisermos as relações $\frac{dx}{d\xi}$ ou $\frac{d\xi}{dx}$ fica mais fácil de obter, de fato a expressão:

$$\frac{dx}{d\xi} = \frac{x_e^+ - x_e^- }{2} = J$$

sendo $J$ o Jacobiano da transformação, e 

$$\frac{d\xi}{dx} = \frac{2}{x_e^+ - x_e^-} = \frac{1}{J}$$

Agora perceba... o que acabamos de fazer é pensando em um único elemento, ou seja, o Jacobiano de um elemento no caso unidimensional é a metade da diferença entre a coordenada **"nó da frente"** $VX(Nv+1)$  com a coordenada **"nó de trás"** $VX(N)$. 

Então para o caso de fazer operações com mais de um elemento (e faremos) podemos calcular o Jacobiano de todos de forma "automática" simplesmente usando a varredura dos vetores:
```Python
J = 0.5*(VX[1:]-VX[:-1])
```

veja que para o coeficiente linear que aparece na transformação, ao qual chamaremos de $B$, a ideia é similar:
```Python
B = 0.5*(VX[1:]+VX[:-1])
```

---

Podemos aproveitar o cálculo do Jacobiano e ainda obter todas as coordenadas de cada elemento no domínio global associadas aos pontos de quadratura $(\xi_i)$! Essa informação será muito útil quando formos montar:
* O **TimeStep** da integração numérica
* Elucidar o **Fluxo** real do problema
* Definir estratégias de *slope limiters*

---

**Vamos fazer um exemplo!**

#### **Exemplo de criação das coordenadas globais devido à quadratura**

Considere o caso em que, para o domínio real $[0, 1]$, definimos $K = 2$ elementos e um polinômio aproximador de grau $P = 2$. 

Como discutido na **Aula 05**, precisaremos de $P+1$ pontos de quadratura (ou seja, 3 pontos) para integrar o nosso polinômio de base. Para retornar esses 3 pontos considerando a quadratura de Gauss-Legendre, podemos informar à nossa função o grau exato $P_{exato} = 2P$.

In [68]:
P = 2 # Grau do Polinômio aproximador
K = 2 # Número de elementos da malha
xmin = 0.0
xmax = 1.0
# Pontos e pesos para quadratura
xi, wi = jacobi_gauss_quad(2*P, quad_type='GL')  
print(f"Qtd de pontos de quadratura: {len(xi)}")

# Geração da malha
Nv, VX, EToV = MeshGen1D(K, xmin, xmax)
print(f"Coordenadas da malha original: {VX}")

Qtd de pontos de quadratura: 3
Coordenadas da malha original: [0.  0.5 1. ]


Note que as coordenadas originais são $VX = [0.0, 0.5, 1.0]$, sendo o intervalo $[0.0, 0.5]$ pertencente ao **elemento 1** e $[0.5, 1.0]$ ao **elemento 2**. Precisamos então mapear os três pontos internos relacionados à quadratura GL para dentro de cada um desses elementos.

> **Nota conceitual:** Se estivéssemos utilizando a quadratura GLL neste caso, a solução seria visualmente mais intuitiva, pois teríamos exatamente os nós das fronteiras da malha e a coordenada central a eles! 

---

> **Nota metodológica:** Existem diferentes estruturas de dados para armazenar os resultados que procuramos. Para manter a filosofia de organização do professor Alberto, nós exigiremos uma saída na qual os nós internos de um mesmo elemento fiquem alocados na mesma **coluna**. Isso pode ser visualizado na tabela ilustrativa abaixo:

<div align="center">

| | **Elemento 1** | **Elemento 2** |
|:-------:|:-----:|:-----:|
|$x^-$      | 0.0 | 0.5|
|$x(\xi_1)$ |     |    |
|$x(\xi_2)$ |     |    |
|$x(\xi_3)$ |     |    |
|$x^+$      | 0.5 | 1.0|

</div>

---
 
Para nos ajudar na obtenção das coordenadas globais dos nós de integração de cada elemento, vamos recorrer à fórmula de mapeamento isoparamétrico que deduzimos:

$$x_{e}(\xi) = \frac{x_e^+ - x_e^- }{2}\xi + \frac{x_e^- + x_e^+}{2} = J \xi + B$$

In [69]:
J = 0.5*(VX[1:] - VX[:-1])
print(f"J: {J}")
B = 0.5*(VX[1:] + VX[:-1])
print(f"B: {B}")
print(f"xi: {xi}")

J: [0.25 0.25]
B: [0.25 0.75]
xi: [-0.77459667  0.          0.77459667]


Como temos $2$ elementos e $3$ nós internos por elemento, a nossa matriz resultante (que chamaremos de `xcoord`) deve ter o formato $3 \times 2$. Vamos avaliar o que temos em mãos. No momento, o nosso código retorna os seguintes **vetores**:

$$\xi_{1 \times 3} \hspace{2cm} J_{1 \times 2} \hspace{2cm} B_{1 \times 2}$$

Matematicamente, para obter a matriz completa de uma só vez, deveríamos operar fazendo um produto externo somado ao coeficiente linear:

$$x^{\text{coord}}_{3 \times 2} = \xi_{3 \times 1} J_{1 \times 2} + \mathbb{1}_{3 \times 1} B_{1 \times 2}$$

> **Desafio Computacional:** Lembre-se de que estamos programando em Python! O NumPy, por padrão, gera vetores unidimensionais com formato `(N,)` (por exemplo, `(3,)` ou `(2,)`), o que significa que uma das dimensões "não existe" para o interpretador. Para replicar a matemática acima sem usar laços de repetição (`for`), podemos contornar esse comportamento utilizando a função `reshape`, a regra *broadcasting* ou `outer`.

**Usando Reshape**

In [60]:
xi1 = xi.reshape(-1,1)
J1 = J.reshape(1,-1)
B1 = B.reshape(1,-1)
xcor1 = xi1 @ J1 + B1
xcor1

array([[0.05635083, 0.55635083],
       [0.25      , 0.75      ],
       [0.44364917, 0.94364917]])

**Usando Broadcasting**

In [63]:
xcor2 = xi[:, None] * J[None, :] + B[None, :]
xcor2

array([[0.05635083, 0.55635083],
       [0.25      , 0.75      ],
       [0.44364917, 0.94364917]])

**Usando Outer**

In [64]:
xcor3 = np.outer(xi, J) + B
xcor3

array([[0.05635083, 0.55635083],
       [0.25      , 0.75      ],
       [0.44364917, 0.94364917]])

### **Codando**

In [7]:
def Jacobian(xi: np.ndarray, VX: np.ndarray):
    """
    Computes the Jacobian and maps quadrature points from the
    reference element [-1,1] to every physical element.

    Parameters
    ----------
    xi : ndarray (Nq,)
        Quadrature points in the reference element.
    VX : ndarray (K+1,)
        Coordinates of the mesh vertices.

    Returns
    -------
    xcoord : ndarray (Nq, K)
        Physical coordinates of every quadrature point in every element.
    J : ndarray (K,)
        Jacobian of each element.
    """
    J = 0.5 * (VX[1:] - VX[:-1])
    B = 0.5 * (VX[1:] + VX[:-1])
    xcoord = xi[:, None] * J[None, :] + B[None, :]

    return xcoord, J

**Exemplo completo**
* Avaliar com maior grau do polinomio aproximador
* Trocar a quantidade de elementos
* Trocar o dominio global
* Trocar a quadratura

In [8]:
P1 = 2 # Grau do Polinomio aproximador
K1 = 5 # numero de elementos para a malha
xmin1 = 0.0
xmax1 = 2.0
xi1, wi1 = jacobi_gauss_quad(2*P1,quad_type='GL')  # Pontos e pesos para quadratura
print(f"Qtd pontos quadratura: {len(xi1)}")
Nv1, VX1, EToV1 = MeshGen1D(K1,xmin1,xmax1)
print(f"Coordenadas da malha original: {VX1}")
xcoord1, jacobianos1 = Jacobian(xi1,VX1)
print("\nCoordenadas internas devido aos pontos da quadratura")
print(xcoord1)
print(f"\nJacobiano de cada elemento:{jacobianos1}")


Qtd pontos quadratura: 3
Coordenadas da malha original: [0.  0.4 0.8 1.2 1.6 2. ]

Coordenadas internas devido aos pontos da quadratura
[[0.04508067 0.44508067 0.84508067 1.24508067 1.64508067]
 [0.2        0.6        1.         1.4        1.8       ]
 [0.35491933 0.75491933 1.15491933 1.55491933 1.95491933]]

Jacobiano de cada elemento:[0.2 0.2 0.2 0.2 0.2]


## **Base Legendre**


Em diversos momentos da resolução de um problema com DG, precisamos avaliar as funções de base nos pontos da malha ou do elemento. Vimos que utilizar os polinômios de Jacobi é uma excelente opção. No entanto, a partir de agora, fixaremos a nossa base como sendo a de **Legendre** ($\alpha = 0, \beta = 0$).

Além disso, pela própria construção do método, operadores fundamentais (como a Matriz de Massa e a Matriz de Rigidez) exigem que essa avaliação seja feita para todos os graus, variando do grau $0$ até a ordem $P$ do nosso polinômio aproximador. Desta maneira, se faz necessário definir uma rotina computacional que construa essa matriz de forma automática.

---

**Exemplo Rápido**  
Se escolhermos o aproximador com grau $P = 2$, precisaremos que a função avalie **3 pontos** da quadratura, utilizando os polinômios de base $P_0^{0,0}, P_1^{0,0}, P_2^{0,0}$. Isso deve nos gerar uma matriz com a seguinte estrutura:

$$
\Phi = \begin{bmatrix}
P_0(\xi_0) & P_1(\xi_0) & P_2(\xi_0) \\
P_0(\xi_1) & P_1(\xi_1) & P_2(\xi_1) \\
P_0(\xi_2) & P_1(\xi_2) & P_2(\xi_2) \\
\end{bmatrix}
$$

#### **Codando**

In [4]:
import numpy as np

def LegendreBasis(P: int, xi: np.ndarray) -> np.ndarray:
    """
    Constrói a matriz de avaliação da base ortogonal de Legendre.

    Parâmetros:
        P (int): Grau máximo do Polinômio Aproximador.
        xi (array): Vetor com os pontos (coordenadas) a serem avaliados.

    Retorna:
        Lj (ndarray): Matriz (npts, P+1) contendo as funções de base de 
                      grau 0 a P avaliadas nos pontos xi.
    """
    Q = P + 1        # Número total de funções de base
    npts = len(xi)   # Número de pontos de avaliação
    
    # Aloca a matriz para armazenar a base de Legendre
    Lj = np.zeros((npts, Q)) 
    
    # Avaliação da base de Legendre (alpha=0, beta=0)
    for j in range(Q):          # Itera sobre o grau do polinômio (0 até P)
        for i in range(npts):   # Itera sobre os pontos da quadratura
            Lj[i, j] = JacobiP(xi[i], j, 0, 0)
            
    return Lj

In [6]:
Pp = 4 # Grau do Polinomio aproximador
xip, wip = jacobi_gauss_quad(2*Pp,quad_type='GL')
psi = LegendreBasis(Pp,xip)
print("Matriz dos Polinômios de Legendre")
psi

Matriz dos Polinômios de Legendre


array([[ 1.        , -0.90617985,  0.73174287, -0.50103117,  0.24573546],
       [ 1.        , -0.53846931, -0.0650762 ,  0.4173821 , -0.34450089],
       [ 1.        ,  0.        , -0.5       , -0.        ,  0.375     ],
       [ 1.        ,  0.53846931, -0.0650762 , -0.4173821 , -0.34450089],
       [ 1.        ,  0.90617985,  0.73174287,  0.50103117,  0.24573546]])

# **Matrizes de Massa e Rigidez**

Antes prosseguirmos para a criação das Matrizez de **Massa** e **Rigidez**, vamos relembrar como elas surgem. Para isso começaremos avaliando um problema "puramente advectivo" (ou convectivo). 

> **Problemas Puramente Advectivos**   
Na física e na engenharia, a advecção é o mecanismo de transporte de uma quantidade (como massa, calor ou quantidade de movimento) puramente pelo movimento global de um fluido.  
Diferente da difusão, que espalha e suaviza as distribuições ao longo do tempo (como uma gota de tinta na água), a advecção pura transporta a informação a uma determinada velocidade sem borrar a sua forma. Matematicamente, esses problemas são modelados por equações diferenciais parciais hiperbólicas de primeira ordem.

A forma mais geral de uma lei de conservação unidimensional puramente advectiva é dada por:

$$\dfrac{\partial u}{\partial t} + \dfrac{\partial f(u)}{\partial x} = 0$$

em $u(x,t)$ é a variável conservativa (como densidade ou velocidade) e $f(u)$ é a função de fluxo. A complexidade do nosso *solver* mudará radicalmente a depender da natureza de $f(u)$:
* Se $f(u) = a u$, temos a **Equação da Advecção Linear**, na qual a onda viaja com velocidade constante $a$ e sua forma permanece inalterada.
* Se $f(u) = \frac{u^2}{2}$, recaímos na famosa **Equação de Burgers Invíscida (Inviscid Burgers)**, um problema não-linear onde a velocidade de propagação depende da própria solução. Isso causa o enrijecimento da onda e a inevitável formação de choques.

### **Forma Fraca e o surgimento dos Operadores Espaciais**

Para que o computador consiga resolver a nossa lei de conservação  
$$\dfrac{\partial u}{\partial t} + \dfrac{\partial f(u)}{\partial x} = 0$$
precisamos transformá-la do seu formato contínuo (forma forte) para um formato matricial (forma fraca local). 

Para isso aplicamos a técnica de Galerkin em cada elemento de forma isolada, criando os nossos operadores espaciais.

---
**Passo 1: Multiplicação pela função teste e integração**  
O primeiro passo é isolar um elemento genérico da malha, $\Omega_k = [x_L, x_R]$. 

Multiplicamos nossa equação original por uma função de teste $\phi_j(x)$ pertencente à nossa base polinomial e integramos sobre o domínio do elemento:

$$\int_{x_L}^{x_R} \frac{\partial u}{\partial t} \phi_j(x) dx + \int_{x_L}^{x_R} \frac{\partial f(u)}{\partial x} \phi_j(x) dx = 0 \tag{1}$$

---
**Passo 2: Integração por partes**  
A grande sacada matemática do DG é aplicar a integração por partes no termo do fluxo. Isso nos permite "jogar" a derivada espacial para cima da função de teste, aliviando a exigência de suavidade da solução original e gerando os termos de fronteira que conectarão os elementos. 

> **Integral por Partes**  
De maneira geral, uma integral por partes é definida como: $$\int_a^b u dv = [uv]_a^b - \int_a^b v du$$  

Se chamarmos $u = \phi_j(x)$ e $dv = \frac{\partial f(u)}{\partial x}dx$ então:
$$du = \frac{d\phi_j(x)}{dx}dx  \hspace{2cm}  v = f(u) $$
e assim
$$\int_{x_L}^{x_R} \frac{\partial f(u)}{\partial x} \phi_j(x) dx = [\phi_j(x)f(u)]_{x_L}^{x_R} - \int_{x_L}^{x_R} f(u)\frac{d\phi_j(x)}{dx}dx $$

subtituindo esse resultado em (1), obtemos:

$$\boxed{\int_{x_L}^{x_R} \phi_j(x) \frac{\partial u}{\partial t}  dx - \int_{x_L}^{x_R} f(u) \frac{d\phi_j(x)}{dx} dx + \left[ f(u)\phi_j(x) \right]_{x_L}^{x_R} = 0} \tag{2}$$

Note que a equação agora revela as três peças fundamentais que precisamos programar no nosso *solver*:
1. **Termo temporal (Matriz de Massa):** Envolve a derivada da solução no tempo e a inércia do elemento.
2. **Termo interno (Matriz de Rigidez):** É a integral negativa que contém o fluxo avaliado no volume do elemento acoplado à derivada da função de base.
3. **Termo de fronteira (Fluxos Numéricos):** Avalia a solução nos limites $x_L$ e $x_R$ conectando a malha global.

---
**Conexão com o Domínio de Referência**  
Se fizermos a mudança de coordenadas do domínio físico $x$ para o domínio de referência $\xi$ (utilizando o nosso Jacobiano $J$) e expandirmos a solução na nossa base $u(x,t)$, 

$$u(x,t) = \sum_{i=0}^P c_i(t)\phi_i({\xi(x)})$$

recuperamos exatamente a Equação 3 apresentada na **Aula 00**. 

Observe como os três blocos que deduzimos acima se traduzem perfeitamente na forma matricial que iremos codificar:

$$
\underbrace{J_e \left[\int_{-1}^{1} \phi_i \phi_j d \xi \right]}_{\text{Jacobiano e Matriz de Massa} \ \mathcal{M}}
\dfrac{\partial}{\partial t} \begin{Bmatrix} c_0 \\ c_1 \\ \vdots \\ c_P \end{Bmatrix} =
\underbrace{\int_{-1}^{1} f \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi }_{\text{Fluxo, Matriz de Derivação} \ \mathcal{D} \ \text{e Rigidez} \ \mathcal{S}} -
\underbrace{\tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} +
\tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix}}_{\text{Fluxos nas Fronteiras}}
$$

---

A partir de agora, vamos isolar e transformar cada uma dessas integrais analíticas em código Python, começando pela avaliação da **Matriz de Massa** e, em seguida, explorando as **Matrizes de Rigidez**.


## **Matriz de Massa**

### **Dedução da Matriz de Massa**

Da equação (2) o termo que chamamos de **Matriz de Massa** ($\mathcal{M}$) sai do agrupamento geométrico que acompanha o termo da derivada temporal, isto é, ele surge do trecho:

$$\int_{x_L}^{x_R} \phi_j(x) \frac{\partial u}{\partial t} dx$$

Vamos fazer a dedução para ver tal matriz surgir similarmente a expressão da equação (3) da **Aula 00**. Começamos com a suposição similar a Moura (2011) em que:

$$u(x,t) = \sum_{i=0}^P c_i(t)\phi_i(x)$$

e disso
$$\frac{\partial u}{\partial t} = \frac{\partial }{\partial t} \sum_{i=0}^P c_i(t)\phi_i(x) = \sum_{i=0}^P \phi_i(x) \frac{\partial c_i(t)}{\partial t} = \{\phi_0, \phi_1, \dots , \phi_P \} \frac{\partial}{\partial t} \begin{Bmatrix}c_0 \\ c_1 \\ \vdots \\ c_P\end{Bmatrix}$$

se substituirmos esse resultado na nossa integral

$$\int_{x_L}^{x_R} \phi_j(x) \{\phi_0, \phi_1, \dots , \phi_P \} \frac{\partial}{\partial t} \begin{Bmatrix}c_0 \\ c_1 \\ \vdots \\ c_P\end{Bmatrix} dx = \int_{x_L}^{x_R} \{\phi_j(x)\phi_0, \phi_j(x)\phi_1, \dots ,\phi_j(x) \phi_P \} dx \frac{\partial}{\partial t} \begin{Bmatrix}c_0 \\ c_1 \\ \vdots \\ c_P\end{Bmatrix}  $$

ou de forma compacta
$$\int_{x_L}^{x_R} \phi_i \phi_j dx \frac{\partial}{\partial t} \begin{Bmatrix}c_0 \\ c_1 \\ \vdots \\ c_P\end{Bmatrix}$$

Como já estudamos nas aulas de quadraturas e também com base na literatura, é muito mais interessante resolvermos integrais em um domínio padrão $\xi\in[-1,1]$ do que no domínio real $x \in [x_L,x_R]$ então devemos fazer a mudança de coordenadas no trecho anterior, vamos usar a mesma estratégia da seção sobre o Jacobiano o que nos dará:

$$\int_{x_L}^{x_R} \phi_i \phi_j dx = \int_{-1}^{1} \phi_i \phi_j \frac{dx}{d \xi} d \xi = J_e \int_{-1}^{1} \phi_i(\xi) \phi_j(\xi) d \xi$$

e essa integral que multiplica o Jacobiano é a nossa matriz de massa

$$\mathcal{M}_{ij} = \int_{-1}^1 \phi_i(\xi) \phi_j(\xi) d\xi$$

---

> Note que $$\int_{x_L}^{x_R} \phi_j(x) \frac{\partial u}{\partial t} dx \Longrightarrow J_e \mathcal{M}_{ij} \frac{\partial}{\partial t} \begin{Bmatrix}c_0 \\ c_1 \\ \vdots \\ c_P\end{Bmatrix}$$  
nós iremos conseguir isolar a derivada temporal passando $J_e$ e $\mathcal{M}_{ij}$ para o lado esquerdo da equação futuramente!

---

### **Calculando a Matriz de Massa**

Em seções anteriores, nós fixamos a nossa função de aproximação usando como base os **Polinômios de Legendre**. Então, podemos calcular as componentes de $\mathcal{M}_{ij}$ tomando a mesma base e ainda utilizar a ideia de Quadraturas ao nosso favor:

$$\mathcal{M}_{ij} = \int_{-1}^1 P^{0,0}_i(\xi) P^{0,0}_j(\xi) d \xi \approx \sum_{m=0}^{Q-1} w_m P^{0,0}_i(\xi_m) P^{0,0}_j(\xi_m)$$

Porém, esse é o exato formato do **Exemplo 6** da **Aula 03**, no qual, com base na teoria de polinômios ortogonais, apresentamos que:

> *Se as funções de aproximação são **Polinômios Ortogonais**, o resultado dessa integral será definido pela delta de Kronecker* $\delta_{ij}$. *No caso de utilizarmos os **Polinômios de Legendre**, o resultado será a $\delta_{ij}$ multiplicada por uma ponderação analítica:*

$$\mathcal{M}_{ij} = \int_{-1}^1 P_i(\xi) P_j(\xi) d \xi = \dfrac{2}{2j+1}\delta_{ij}$$

Ou seja, a integral obedece à regra:

$$\mathcal{M}_{ij} = 
\begin{cases} 
0 & \text{se } i \neq j \\ 
\dfrac{2}{2j+1} & \text{se } i = j 
\end{cases}$$

Consequentemente, a nossa **Matriz de Massa** será estritamente **diagonal**. Isto indica que, para criar nosso código, podemos definir/usar duas estratégias:

$$\boxed{\mathcal{M}_{jj} =\sum_{m=0}^{Q-1} w_m (P^{0,0}_j(\xi_m))^2 = \dfrac{2}{2j+1}}$$

---

> Em métodos numéricos tradicionais (onde a matriz é cheia e densa), inverter a Matriz de Massa para avançar no tempo é uma operação computacionalmente muito cara. No nosso *solver* DG 1D, como a matriz é diagonal, a sua inversa ($\mathcal{M}^{-1}$) é obtida trivialmente invertendo os elementos da própria diagonal:

$$\boxed{\mathcal{M}^{-1}_{ii} = \frac{1}{\mathcal{M}_{ii}}}$$

#### **Codando**

In [15]:
import numpy as np

def MassMatrix(Nldof: int, method: str = 'analytic', wi: np.ndarray = None, psi: np.ndarray = None):
    """
    Constrói a diagonal da Matriz de Massa e sua respectiva Inversa.
    
    Parâmetros:
        Nldof (int): Número de graus de liberdade locais (Grau P + 1). Obrigatório.
        method (str): 'analytic' (padrão) ou 'quadrature'.
        wi (array): Pesos da quadratura. Obrigatório se method='quadrature'.
        psi (ndarray): Base de Legendre avaliada. Obrigatório se method='quadrature'.

    Retorna:
        diagM (array): Vetor contendo a diagonal principal da Matriz de Massa.
        invDiagM (array): Vetor contendo a inversa da diagonal principal.
    """
    diagM = np.zeros(Nldof)
    invDiagM = np.zeros(Nldof)
    
    if method == 'analytic':
        for n in range(Nldof):
            # Solução exata da integral do Polinômio de Legendre ao quadrado
            diagM[n] = 2.0 / (2.0 * n + 1.0)
            invDiagM[n] = 1.0 / diagM[n]
            
    elif method == 'quadrature':
        # Trava de segurança: verifica se os dados foram passados
        if wi is None or psi is None:
            raise ValueError("Erro: Para o método 'quadrature', você deve fornecer os argumentos 'wi' e 'psi'!")
            
        for i in range(Nldof):
            # Solução via integração numérica
            diagM[i] = np.sum(wi * (psi[:, i]**2))
            invDiagM[i] = 1.0 / diagM[i]
            
    else:
        raise ValueError("Erro: Método desconhecido. Escolha 'analytic' ou 'quadrature'.")
        
    return diagM, invDiagM

**Opção 1**

In [16]:
Pdg = 2
Ndof = Pdg + 1
Mij2, invMij2 = MassMatrix(Ndof)

print("Matriz de Massa")
print(Mij2)
print("Inversa da Matriz de Massa")
print(invMij2)

Matriz de Massa
[2.         0.66666667 0.4       ]
Inversa da Matriz de Massa
[0.5 1.5 2.5]


**Opção 2** 

In [18]:
Pdg = 2
Ndof = Pdg + 1
xx, ww = jacobi_gauss_quad(2*Pdg)
psiu = LegendreBasis(Pdg,xx)

Mij1, invMij1 = MassMatrix(Ndof,method='quadrature',wi = ww,psi = psiu)

print("Matriz de Massa")
print(Mij1)
print("Inversa da Matriz de Massa")
print(invMij1)

Matriz de Massa
[2.         0.66666667 0.4       ]
Inversa da Matriz de Massa
[0.5 1.5 2.5]


## **Matriz de Rigidez**

A matriz de Rigidez pode ser classificada em dois tipos:

* Chamamos de **Matriz de Rigidez Tradicional** aquela atrelada a um formalismo que contém puramente termos **advectivos/convectivos**.

* Chamamos de **Matriz de Rigidez Modificada** a matriz necessária para resolver problemas que possuem **difusividade**. Essa difusão pode ser uma característica real do escoamento ou advinda de uma estratégia numérica com a "adição" de uma **viscosidade artificial** para estabilizar o método.

### **Dedução da Matriz de Rigidez Tradicional**

A **Matriz de Rigidez Tradicional** atua diretamente sobre os fluxos que transportam informação ao longo do domínio. O nosso foco agora é isolar o termo interno da forma fraca e transformá-lo na matriz de rigidez computacional, ou seja nosso termo de interesse é:

$$\int_{x_L}^{x_R} f(u) \frac{d\phi_j(x)}{dx} dx$$

Como já vimos para a dedução da matriz de massa, é mais interessante resolver a integral no domínio de referência padrão $\xi \in [-1, 1]$ do que no domínio real $\Omega_k$. Então precisamos mapear essa integral. Lembrando que o diferencial geométrico é $\color{blue}{dx = J d\xi}$, onde $J = \frac{h_k}{2}$ é o Jacobiano (metade do tamanho do elemento). Mas atenção ao aplicar a regra da cadeia na derivada da função de base:

$$\frac{d\phi_j}{dx} = \frac{d\phi_j}{d\xi} \frac{d\xi}{dx} = {\color{red}{\frac{1}{J}}} \frac{d\phi_j}{d\xi}$$

Quando substituímos tudo isso na nossa integral o Jacobiano multiplicando o diferencial ($J d\xi$) e o Jacobiano dividindo a derivada ($1/J$) se cancelam perfeitamente! A integral independente do tamanho real do elemento torna-se:

$$\int_{-1}^1 f(u(\xi)) {\color{red}{\frac{1}{J}}} \frac{d\phi_j}{d\xi} {\color{blue}{J d\xi}} = \int_{-1}^1 f(u(\xi)) \frac{d\phi_j(\xi)}{d\xi} d\xi$$

Precisamos agora nos livramos do fluxo, visto que, queremos um operador puramente geométrico e polinomial. Para isso, vamos fazer com que esse fluxo físico seja **projetado** pela nossa base polinomial de Jacobi/Legendre, criando um vetor de coeficientes modais de fluxo, que chamaremos de $\hat{f}_i$:

$$f(u(\xi)) \approx \sum_{i=0}^{N} \hat{f}_i \phi_i (\xi)$$

substituindo isso na integral

$$\int_{-1}^1 \sum_{j=0}^{N} \hat{f}_i \phi_i (\xi) \frac{d\phi_j(\xi)}{d\xi} d\xi \Longrightarrow  \sum_{i=0}^{N} \hat{f}_i \left( \int_{-1}^1 \phi_i (\xi) \frac{d\phi_j(\xi)}{d\xi} d\xi \right)$$
note que agora podemos substituír a integral analítica pela quadratura de Gauss.

Se usarmos os pontos de integração $\xi_m$ e os pesos $w_m$ desenvolvidos na **Aula 03** temos:

$$\sum_{i=0}^{N} \hat{f}_i \left( \int_{-1}^1 \phi_i (\xi) \frac{d\phi_j(\xi)}{d\xi} d\xi \right) = \sum_{i=0}^{N} \hat{f}_i \left( \sum_{m=0}^{Q-1} w_m \phi_i(\xi_m) \frac{d\phi_j(\xi_m)}{d\xi} \right)$$

o termo dentro do parênteses é a nossa Matriz de Rigidez $\mathcal{S}$ associada às interações entre os modos $i$ e $j$:

$$\mathcal{S}_{ij} = \sum_{m=0}^{Q-1} w_m \phi_i(\xi_m) \frac{d\phi_j(\xi_m)}{d\xi}$$

---

**NOTAS**
> Nesta aula não iremos nos preocupar com o que acontece com o fluxo, porém de fato ele não pode ser ignorado; ele irá retornar quando estudarmos sobre **Projeção de Fluxos** e a construção do **Operador Lh** em aulas subsequentes.

---




**Exemplo Detalhado**

Considere que escolhemos um polinômio de grau $P = 2$ para ser nosso aproximador, como vimos na **aula 05** isso indicar que precisamos de $2P+1$ como o grau exato para encontrarmos os valores $w_i$ e $\xi_i$ da quadratura.

No caso de $P = 2$ o valor grau exato se torna $5$ e esse é exato para $3$ pontos (esse é o valor de $Q$ no somatório da matriz de rigidez!!)

---
```Python  
P_max = 2     # Escolher o grau igual 2
q_type='GL'   # Explicitando a quadratura
xGL, wi = jacobi_gauss_quad(2*P_max+1, quad_type=q_type)
```

---

esse trecho nos retorna $\xi_0, \xi_1, \xi_2, w_0, w_1$ e $w_2$.

A definição da matriz de rigidez como definimos é:

$$\mathcal{S}_{ij} = \sum_{m=0}^{2} w_m \phi_i(\xi_m) \phi'_j(\xi_m)$$

observe que, como escolhemos um polinômio de grau P=2, existem três funções de base. Assim, os índices variam entre 0 e 2, isto é, $0\leq i,j \leq 2$. Expandindo cada um dos 9 elementos da matriz (que Zeus nos ajude 🥲), obtemos:

$$S_{00} = w_0 \phi_0(\xi_0) \phi'_0(\xi_0) + w_1 \phi_0(\xi_1) \phi'_0(\xi_1) + w_2 \phi_0(\xi_2) \phi'_0(\xi_2)$$
$$S_{01} = w_0 \phi_0(\xi_0) \phi'_1(\xi_0) + w_1 \phi_0(\xi_1) \phi'_1(\xi_1) + w_2 \phi_0(\xi_2) \phi'_1(\xi_2)$$
$$S_{02} = w_0 \phi_0(\xi_0) \phi'_2(\xi_0) + w_1 \phi_0(\xi_1) \phi'_2(\xi_1) + w_2 \phi_0(\xi_2) \phi'_2(\xi_2)$$

$$S_{10} = w_0 \phi_1(\xi_0) \phi'_0(\xi_0) + w_1 \phi_1(\xi_1) \phi'_0(\xi_1) + w_2 \phi_1(\xi_2) \phi'_0(\xi_2)$$
$$S_{11} = w_0 \phi_1(\xi_0) \phi'_1(\xi_0) + w_1 \phi_1(\xi_1) \phi'_1(\xi_1) + w_2 \phi_1(\xi_2) \phi'_1(\xi_2)$$
$$S_{12} = w_0 \phi_1(\xi_0) \phi'_2(\xi_0) + w_1 \phi_1(\xi_1) \phi'_2(\xi_1) + w_2 \phi_1(\xi_2) \phi'_2(\xi_2)$$

$$S_{20} = w_0 \phi_2(\xi_0) \phi'_0(\xi_0) + w_1 \phi_2(\xi_1) \phi'_0(\xi_1) + w_2 \phi_2(\xi_2) \phi'_0(\xi_2)$$
$$S_{21} = w_0 \phi_2(\xi_0) \phi'_1(\xi_0) + w_1 \phi_2(\xi_1) \phi'_1(\xi_1) + w_2 \phi_2(\xi_2) \phi'_1(\xi_2)$$
$$S_{22} = w_0 \phi_2(\xi_0) \phi'_2(\xi_0) + w_1 \phi_2(\xi_1) \phi'_2(\xi_1) + w_2 \phi_2(\xi_2) \phi'_2(\xi_2)$$

Por mais que não seja tão trivial saber qual o formato exato, mas sabemos que de alguma forma isso pode ser escrito como a multiplicação de matrizes, uma relacionada a função e outra a derivada da função. Vamos assumir que a matriz de funções é dada de forma similar ao que vimos da construção para a **base do Polinômio de Legendre** (o grau do polinomio aumenta com as colunas e o ponto avaliado com as linhas):

$$
\Phi = \begin{bmatrix}
\phi_0(\xi_0) & \phi_1(\xi_0) & \phi_2(\xi_0) \\
\phi_0(\xi_1) & \phi_1(\xi_1) & \phi_2(\xi_1) \\
\phi_0(\xi_2) & \phi_1(\xi_2) & \phi_2(\xi_2) \\
\end{bmatrix}
$$

e guardamos esse resultado.

---


Vimos tanto na **aula 04** quanto na **05** que a derivada numérica de uma função pode ser calculada com o uso da matriz de Diferenciação $(\mathcal{D}_r)$ como segue:

$$ \frac{df(\xi_m)}{d\xi} = \mathcal{D}_r f(\xi_m) $$

podemos é claro, assumir que essa função é simplesmente dada pelo nosso polinômio aproximador de grau $j$, ao qual chamamos por conveniência de $\phi_j$, ou seja,

$$ \frac{d\phi_j(\xi_m)}{d\xi} = \mathcal{D}_r \phi_j(\xi_m) $$

o resultado dessa derivada, é simplemesmente o vetor de derivadas desse polinômio calculado nos pontos da quadratura, ou seja:

$$\frac{d\phi_j(\xi_m)}{d\xi} = \mathcal{D}_r \phi_j(\xi_m) = \begin{Bmatrix} \phi'_j(\xi_0) \\ \phi'_j(\xi_1) \\ \phi'_j(\xi_2) \end{Bmatrix}$$

então como nosso formalismo até aqui nos permite calcular a derivada numérica para uma função $\phi_j$ conseguiremos obter uma matriz com derivadas para $\phi_0, \phi_1$ e $\phi_2$ o que segue:

$$
\Phi' = \begin{bmatrix}
\phi'_0(\xi_0) & \phi'_1(\xi_0) & \phi'_2(\xi_0) \\
\phi'_0(\xi_1) & \phi'_1(\xi_1) & \phi'_2(\xi_1) \\
\phi'_0(\xi_2) & \phi'_1(\xi_2) & \phi'_2(\xi_2) \\
\end{bmatrix}
$$

---

Vamos colocar as duas matrizes lado a lado

$$
\Phi = \begin{bmatrix}
\phi_0(\xi_0) & \phi_1(\xi_0) & \phi_2(\xi_0) \\
\phi_0(\xi_1) & \phi_1(\xi_1) & \phi_2(\xi_1) \\
\phi_0(\xi_2) & \phi_1(\xi_2) & \phi_2(\xi_2) \\
\end{bmatrix}  \hspace{2cm}
\Phi' = \begin{bmatrix}
\phi'_0(\xi_0) & \phi'_1(\xi_0) & \phi'_2(\xi_0) \\
\phi'_0(\xi_1) & \phi'_1(\xi_1) & \phi'_2(\xi_1) \\
\phi'_0(\xi_2) & \phi'_1(\xi_2) & \phi'_2(\xi_2) \\
\end{bmatrix}
$$

e olharmos para o que são os termos da primeira linha de $\mathcal{S}$ isto é $\mathcal{S}_{0j}$

$${\color{red}{S_{00}}} = w_0 \phi_0(\xi_0) \phi'_0(\xi_0) + w_1 \phi_0(\xi_1) \phi'_0(\xi_1) + w_2 \phi_0(\xi_2) \phi'_0(\xi_2)$$
$$S_{01} = w_0 \phi_0(\xi_0) \phi'_1(\xi_0) + w_1 \phi_0(\xi_1) \phi'_1(\xi_1) + w_2 \phi_0(\xi_2) \phi'_1(\xi_2)$$
$$S_{02} = w_0 \phi_0(\xi_0) \phi'_2(\xi_0) + w_1 \phi_0(\xi_1) \phi'_2(\xi_1) + w_2 \phi_0(\xi_2) \phi'_2(\xi_2)$$

Note que se multiplicamos a primeira linha de $\Phi$ com a primeira coluna de $\Phi'$ não temos $S_{00} $ mas se fizermos $\Phi^T$ obtemos o resultado que queremos!!

$$\Phi^T\Phi' = \begin{bmatrix}
{\color{red}{\phi_0(\xi_0)}} & {\color{red}{\phi_0(\xi_1)}} & {\color{red}{\phi_0(\xi_2)}}\\
\phi_1(\xi_0) & \phi_1(\xi_1) & \phi_1(\xi_2) \\
\phi_2(\xi_0) & \phi_2(\xi_1) & \phi_2(\xi_2)
\end{bmatrix} \begin{bmatrix}
{\color{red}{\phi'_0(\xi_0)}} & \phi'_1(\xi_0) & \phi'_2(\xi_0) \\
{\color{red}{\phi'_0(\xi_1)}} & \phi'_1(\xi_1) & \phi'_2(\xi_1) \\
{\color{red}{\phi'_0(\xi_2)}} & \phi'_1(\xi_2) & \phi'_2(\xi_2)
\end{bmatrix}$$

Com relação aos pesos $w_0, w_1$ e $w_2$ percebemos que eles aparecem sempre na mesma posição e podem de alguma forma acompanhar a matriz $\Phi^T$ isto é devemos escrever

$$\begin{bmatrix}
w_0 \phi_0(\xi_0) & w_1 \phi_0(\xi_1) & w_2 \phi_0(\xi_2)\\
w_0 \phi_1(\xi_0) & w_1 \phi_1(\xi_1) & w_2 \phi_1(\xi_2) \\
w_0 \phi_2(\xi_0) & w_1 \phi_2(\xi_1) & w_2 \phi_2(\xi_2)
\end{bmatrix} = \begin{bmatrix}
\phi_0(\xi_0) & \phi_0(\xi_1) & \phi_0(\xi_2)\\
\phi_1(\xi_0) & \phi_1(\xi_1) & \phi_1(\xi_2) \\
\phi_2(\xi_0) & \phi_2(\xi_1) & \phi_2(\xi_2)
\end{bmatrix} \begin{Bmatrix} w_0 \\ w_1 \\ w_2 \end{Bmatrix} = \Phi^T \mathbf{w}
$$

assim nossa matriz de rigidez pode ser escrita usando operações matriciais o que segue:

$$\boxed{\mathcal{S} = \Phi^T \mathbf{w} \Phi'}$$

---

> Mediante as ferramentas que contruímos, por conveniência metodológica, a matriz $\Phi$ será gerada via função `LegendreBasis` e $\Phi'$ será obtida com auxílio da função `Dmatrix1D`


#### **Codando**

Podemos definir nossa matriz de Rigidez, para ela precisaremos:
* Dos pesos $w_m$
* Dos polinômios avaliados nos pontos da quadratura $\phi_i(\xi_m)$
* Da matriz contendo as derivadas $\phi'_j(\xi_m)$

In [3]:
def StiffMatrix(wi,phi,Dphi):
    """
    Matriz de Rigidez (Advecção)

    ### Args:
    wi (array): Vetor de pesos da quadratura escolhida
    phi (array): Matriz do Polinômio Base avaliada em xi
    Dphi (array): Matriz de Derivadas do Polinomio Base avaliado em xi

    ### Return
    Retorna a matriz de Rigidez (advecção) do sistema

    """
    #Computes local stiffness matrix
    Sij = np.dot(wi*phi.T,Dphi)

    return Sij

#### Exemplo Numérico

* Precisamos definir o grau do nosso polinomio aproximador
* Definir a quadratura que usaremos
* Encontrar os valores de $\xi$ e $w$
* Avaliar no polinomio base
* Calcular a derivada do polinomio base nesses pontos
* Encontrar a matriz de rigidez

In [5]:
P_max = 2     # Escolher o grau igual 2
q_type='GL'   # Explicitando a quadratura
xGL, wi = jacobi_gauss_quad(2*P_max, quad_type=q_type)

print('Pontos e pesos da quadratura GL')
print(f"xi = {xGL}")
print(f"wi = {wi}")

Pontos e pesos da quadratura GL
xi = [-0.77459667  0.          0.77459667]
wi = [0.55555556 0.88888889 0.55555556]


In [ ]:
Phi = LegendreBasis(P_max,xGL)
Phi

array([[ 1.        , -0.77459667,  0.4       ],
       [ 1.        ,  0.        , -0.5       ],
       [ 1.        ,  0.77459667,  0.4       ]])

In [ ]:
Dr = DMatrix1D(xGL,len(xGL))
DPhi = Dr @ Phi
DPhi

array([[-5.55111512e-16,  1.00000000e+00, -2.32379001e+00],
       [ 0.00000000e+00,  1.00000000e+00, -1.89737731e-17],
       [ 4.44089210e-16,  1.00000000e+00,  2.32379001e+00]])

In [ ]:
Sij = StiffMatrix(wi,Phi,DPhi)
Sij

array([[-6.16790569e-17,  2.00000000e+00, -5.47501540e-17],
       [ 4.29987528e-16,  2.45721145e-17,  2.00000000e+00],
       [-2.46716228e-17,  3.76242247e-16,  1.67988160e-17]])

---

## **Lift Matrices (Matrizes de Elevação)**

Antes de seguirmos para a construção da **Matriz de Rigidez Modificada**, devemos conhecer uma estratégia matemática extremamente importante.

Se olharmos novamente para a nossa forma fraca, perceberemos que o último bloco da equação é dedicado exclusivamente aos termos de fronteira:

$${-\tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} +
\tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix}} = -\phi^+_j \tilde{f}_{e,e+1} + \phi^-_j\tilde{f}_{e-1,e}$$

sendo $\tilde{f}_{e,e+1} = \tilde{f}(u^+_e,u^-_{e+1})$ e $\tilde{f}_{e-1,e} = \tilde{f}(u^+_{e-1},u^-_e)$.

Esses termos são responsáveis por capturar a informação (os **fluxos numéricos**) que viaja das bordas dos elementos vizinhos e "elevá-la" (do inglês, *lift*) para dentro do volume do elemento atual.

Matematicamente, avaliar o termo de fronteira significa avaliar a nossa função de base (a função teste $j$) nos extremos do domínio de referência: na face esquerda ($\xi = -1$) e na face direita ($\xi = +1$), ou seja:

$$-\phi^+_j \tilde{f}_{e,e+1} + \phi^-_j\tilde{f}_{e-1,e} = - \phi_j(+1) \tilde{f}_{e,e+1} + \phi_j(-1)\tilde{f}_{e-1,e}$$

Como escolhemos os **Polinômios de Legendre** como nossa base oficial, nós ganhamos de brinde uma das suas propriedades de contorno mais elegantes:
* Na face direita, todos os polinômios valem $1$: $\quad P_n(1) = 1$
* Na face esquerda, o sinal alterna dependendo se o grau $n$ é par ou ímpar: $\quad P_n(-1) = (-1)^n$

Portanto, a avaliação da nossa função teste nas bordas fica simplificada para:

$$- \phi_j(+1) \tilde{f}_{e,e+1} + \phi_j(-1)\tilde{f}_{e-1,e} = -(1)\tilde{f}_{e,e+1} + (-1)^j\tilde{f}_{e-1,e}$$

### **Onde estão as matrizes?**

#### **Matrizes Locais Frk e Flk**
Para entender como saltamos da equação anterior para as matrizes, precisamos olhar para o que tem dentro do fluxo na fronteira.

Imagine um caso simples onde o fluxo depende do estado $u$ (por exemplo, $\tilde{f} = u$). Sabemos que, no DG, o valor de $u$ em qualquer lugar é a soma dos seus coeficientes modais multiplicados pelas funções de base. Na fronteira direita ($\xi = +1$), isso fica assim:

$$u(+1) = \sum_{m=0}^{P} \hat{u}_m \phi_m(+1)$$

Agora, pegue apenas o primeiro pedaço do nosso termo de fronteira da equação principal, onde temos a função teste $\phi_j$ multiplicando esse fluxo na borda direita:

$$- \phi_j(+1) \cdot u(+1)$$

Se substituirmos a expansão de $u(+1)$ ali dentro, teremos:

$$- \phi_j(+1) \left( \sum_{m=0}^{P} \hat{u}_m \phi_m(+1) \right)$$

Como a função teste $\phi_j(+1)$ é só um número avaliado na borda, podemos "empurrá-la" para dentro do somatório:

$$- \sum_{m=0}^{P} \underbrace{\left[ \phi_j(+1) \phi_m(+1) \right]}_{\text{A Mágica Acontece Aqui!}} \hat{u}_m$$

> Olhe para o termo entre colchetes. Ele depende exclusivamente da geometria da nossa base polinomial! Ele não sabe nada sobre a física do problema ($\hat{u}_m$) nem sobre o tempo. Ele é apenas a interação entre o índice $j$ (da equação teste) e o índice $m$ (da função base) avaliados na borda.

Como $j$ varia de $0$ até $P$ e $m$ varia de $0$ até $P$, essa interação $\left[ \phi_j \phi_m \right]$ gera todas as combinações possíveis, formando uma "tabela" (uma matriz quadrada!). Os elementos dessa matriz são exatamente:

$$\mathcal{L}_{jm} = \phi_j(\text{contorno}) \times \phi_m(\text{contorno})$$

> **Exemplo:**  
Se avaliarmos a interação puramente na face direita ($\xi = +1$), teremos $\mathcal{L}_{jm} = (1) \times (1) = 1$. O resultado é uma matriz inteira preenchida por valores $1$.  
Se avaliarmos puramente na face esquerda ($\xi = -1$), teremos $\mathcal{L}_{jm} = (-1)^j \times (-1)^m$. Os sinais da matriz vão se alternar dependendo se as combinações de linha e coluna são pares ou ímpares.

A essas matrizes criadas uma na face direita e outra na face esquerda denominamos **Frk** e **Flk** respectivamente.



#### **Matrizes de Acoplamento**

O método DG funciona porque os elementos trocam informações através do fluxo numérico.

Imagine que o elemento $k$ está recebendo informação do vizinho da esquerda (o elemento $k-1$). A função teste $\phi_j$ pertence ao elemento $k$ e está "escutando" na borda esquerda ($\xi = -1$). Porém, a informação $u$ que está chegando foi construída usando a base do vizinho $k-1$, que está encostado ali pela sua borda direita ($\xi = +1$).

Quando colocamos isso na equação, o cruzamento que acontece é entre a função teste em $-1$ e a função base do vizinho em $+1$:
$$\mathcal{L}_{jm} = \underbrace{\phi_j(-1)}_{\text{Elemento } k} \times \underbrace{\phi_m(+1)}_{\text{Elemento } k-1}$$

Sabendo as propriedades de Legendre ($P_n(1) = 1$ e $P_n(-1) = (-1)^n$), essa multiplicação se torna:

$$\mathcal{L}_{jm} = (-1)^j \times (1) = (-1)^j$$

Como o sinal $(-1)^j$ depende exclusivamente do índice $j$ (que representa as linhas da nossa matriz), teremos uma matriz onde as **linhas de índice par** são todas positivas e as **linhas de índice ímpar** são todas negativas. A coluna (modo $m$ do vizinho) não altera o sinal! E essa matriz chamamos **Frkm1**!

A mesma lógica se aplica quando o elemento $k$ recebe informação do vizinho da direita ($k+1$). A nossa função teste está na borda direita ($\xi = +1$), e a base do vizinho está na borda esquerda dele ($\xi = -1$):

$$\mathcal{L}_{jm} = \underbrace{\phi_j(+1)}_{\text{Elemento } k} \times \underbrace{\phi_m(-1)}_{\text{Elemento } k+1}$$

$$\mathcal{L}_{jm} = (1) \times (-1)^m = (-1)^m$$

Como o sinal $(-1)^m$ depende exclusivamente do índice $m$ (que representa as colunas da nossa matriz), teremos uma matriz onde as colunas alternam de sinal. E essa matriz chamamos de **Flkp1**!

---

Essa previsibilidade analítica nos permite deixar os operadores de cruzamentos de fronteiras prontos. Nós os chamamos de **Lift Matrices**, e no *solver* 1D teremos as quatro matrizes com dimensão $(N_{ldof} \times N_{ldof})$:

1. `Frk` (Face Right $k$): Interação local $\phi_j(+1) \phi_m(+1)$. Matriz inteiramente preenchida por $1$.
2. `Flk` (Face Left $k$): Interação local $\phi_j(-1) \phi_m(-1)$. O cruzamento altera o sinal da matriz de forma "xadrez".
3. `Frkm1` (Face Right do $k-1$): Acoplamento com o vizinho da esquerda $\phi_j(-1) \phi_m(+1)$. Altera o sinal apenas das linhas ímpares.
4. `Flkp1` (Face Left do $k+1$): Acoplamento com o vizinho da direita $\phi_j(+1) \phi_m(-1)$. Altera o sinal apenas das colunas ímpares.

---

> **O Poder do Python:** Em vez de usarmos laços `for` e condicionais `if` para checar as combinações de $j$ e $m$, podemos simplesmente inicializar matrizes de uns (`np.ones`) e usar o fatiamento matricial (slicing) do NumPy para aplicar os sinais negativos! Por exemplo, para `Flkp1`, usamos `Flkp1[:, 1:Nldof:2] = -1.0` para inverter todas as linhas (`:`) mas apenas das colunas ímpares (`1:Nldof:2`).

In [19]:
import numpy as np

def LiftMatrix(Nldof: int):
    """
    Constrói as Matrizes de Elevação (Lift Matrices) avaliando os Polinômios
    de Legendre nas fronteiras do elemento de referência (xi = -1 e xi = 1).
    
    Parâmetros:
        Nldof (int): Número de graus de liberdade locais (Grau P + 1).
        
    Retorna:
        Flk (ndarray): Matriz da face esquerda do elemento k.
        Frk (ndarray): Matriz da face direita do elemento k.
        Flkp1 (ndarray): Matriz de interação com o vizinho da direita.
        Frkm1 (ndarray): Matriz de interação com o vizinho da esquerda.
    """
    # Aloca as matrizes inicialmente com o valor 1 (assumindo P_n(1) = 1)
    Frk = np.ones((Nldof, Nldof))
    Flk = np.ones((Nldof, Nldof))
    Flkp1 = np.ones((Nldof, Nldof))
    Frkm1 = np.ones((Nldof, Nldof))
    
    # Aplica a propriedade P_n(-1) = (-1)^n
    # O slicing [1:Nldof:2] acessa exatamente as colunas/linhas de grau ímpar (1, 3, 5...)
    Flkp1[:, 1:Nldof:2] = -1.0 * Flkp1[:, 1:Nldof:2]
    Frkm1[1:Nldof:2, :] = -1.0 * Frkm1[1:Nldof:2, :]
    
    # Para a matriz Flk, a alternância ocorre tanto nas linhas quanto nas colunas
    Flk[:Nldof:2, 1:Nldof:2] = -1.0 * Flk[:Nldof:2, 1:Nldof:2]
    Flk[1:Nldof:2, :Nldof:2] = -1.0 * Flk[1:Nldof:2, :Nldof:2]
    
    return Flk, Frk, Flkp1, Frkm1

In [23]:
ndoff = 3
Flk, Frk, Flkp1, Frkm1 = LiftMatrix(ndoff)

print(f"Frk \n {Frk}")

print(f"Flk \n {Flk}")

print(f"Frkm1 \n {Frkm1}")

print(f"Flkp1 \n {Flkp1}")

Frk 
 [[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
Flk 
 [[ 1. -1.  1.]
 [-1.  1. -1.]
 [ 1. -1.  1.]]
Frkm1 
 [[ 1.  1.  1.]
 [-1. -1. -1.]
 [ 1.  1.  1.]]
Flkp1 
 [[ 1. -1.  1.]
 [ 1. -1.  1.]
 [ 1. -1.  1.]]


## **Matriz de Rigidez Modificada**

Até agora, nosso formalismo usou um equacionamento de advecção pura.
$$\frac{\partial u}{\partial t} + ∇ F = 0 \tag{i}$$

No entanto é natural pensar em problemas que contemplem termos de **difusão** (como a condução de calor em um material) [**Warburton**]

$$\frac{\partial u}{\partial t} = \frac{\partial ^2 u}{\partial x^2} \tag{2.1}$$

ou ocasiões no qual **(i)** não é puramente hiperbólico, mas contém uma pequena quantidade de viscosidade e os choques ou descontinuidades tornam-se camadas finas onde a solução muda rapidamente. E neste caso, se faz necessária a adição de **viscosidade artificial** no termo de modelo dissipativo fazendo com que as equações originais tomem a forma [**Persson**]:

$$\frac{\partial u}{\partial t} + ∇ F = \frac{\partial }{\partial x}\left(\epsilon \frac{\partial u}{\partial x}\right) \tag{2.2}$$

Note que, em ambos os casos, surgem derivadas de segunda ordem. Nesse sentido o Método DG tradicional, por usar funções de base descontínuas nas interfaces, sofre para calcular as derivadas segundas de forma direta. Para contornar isso, utilizamos uma técnica chamada **Local Discontinuous Galerkin (LDG)**.

A sacada do LDG é transformar a equação de segunda ordem em um sistema acoplado de duas equações de primeira ordem, introduzindo uma variável auxiliar $q$.

### **Formulações de Persson e Warburton**

A forma como quebramos a equação original em duas define o comportamento e a estabilidade da nossa **Matriz de Rigidez** o que a levará para um modelo **Modificado**. Na literatura, destacam-se duas abordagens:



#### **Abordagem do Warburton (Simétrica e Estável)**  

A abordagem apresentada no livro do Warburton parte da equação (2.1) mas em um formato mais generalizado

$$\frac{\partial u}{\partial t} = \frac{\partial }{\partial x}a(x)\frac{\partial u}{\partial x}$$

Nesta formulação (originária de Cockburn e Shu), a viscosidade $a(x)$ é dividida simetricamente usando a sua raiz quadrada:
$$
q = \sqrt{a(x)} \frac{\partial u}{\partial x} \\
\frac{\partial u}{\partial t} = \frac{\partial}{\partial x}\sqrt{a(x)}q 
$$

Note que agora podemos aplicar a formulação de Galerkin Discontinuo nessas duas equações.

**Passo 1: A equação auxiliar e a Matriz $\mathcal{S}_{sq1}$**  
Pegamos a primeira equação (a variável auxiliar $q$) e aplicamos a formulação de Galerkin: multiplicamos pela função teste $\phi_j(x)$ e integramos sobre o domínio do elemento $\Omega_k$:

$$\int_{\Omega_k} q \phi_j dx = \int_{\Omega_k} \sqrt{a(x)} \frac{\partial u}{\partial x} \phi_j dx$$

A grande sacada do esquema LDG padrão é que não aplicamos integração por partes nesta equação auxiliar. Substituindo a variável física pela sua expansão modal ($u \approx \sum u_i \phi_i$) e mapeando a integral geométrica para o domínio de referência $\xi \in [-1,1]$, o lado direito se consolida na nossa primeira matriz de rigidez modificada:

$$\mathcal{S}_{sq1, ij} = \int_{-1}^{1} \sqrt{a(\xi)} \phi_j \frac{d\phi_i}{d\xi} d\xi$$

> No Código: A viscosidade (chamaremos de `epst`) entra no somatório da quadratura e altera o peso da matriz. Note que o formato é muito similar ao que estruturamos para a matriz tradicional

```Ssq1 = np.dot(np.sqrt(epst)*wi*psi.T, Dpsi)```

**Passo 2: A equação principal e a Integração por Partes**  
Agora olhamos para a segunda equação, a lei de conservação principal:
$$\int_{\Omega_k} \frac{\partial u}{\partial t} \phi_j dx = \int_{\Omega_k} \frac{\partial}{\partial x}\left(\sqrt{a(x)}q\right) \phi_j dx$$

Nesta equação, nós aplicamos a integração por partes no termo da direita para "jogar" a derivada para a função de teste. Isso expõe os limites do elemento:

$$\int_{\Omega_k} \frac{\partial u}{\partial t} \phi_j dx = \left[ \sqrt{a(x)} q \phi_j \right]_{x_L}^{x_R} - \int_{\Omega_k} \sqrt{a(x)} q \frac{d\phi_j}{dx} dx$$


**Passo 3: A simetria revelada ($\mathcal{S}_{sq1}^T$)**  
Observe com atenção a integral interna que sobrou no volume após a integração por partes:$$\int_{\Omega_k} \sqrt{a(x)} q \frac{d\phi_j}{dx} dx$$Se expandirmos o $q$ em nossa base polinomial ($q \approx \sum q_i \phi_i$), a montagem matricial terá a derivada aplicada no índice $j$ (da função teste) e não no índice $i$ (da função de aproximação):

$$\int_{-1}^{1} \sqrt{a(\xi)} \phi_i \frac{d\phi_j}{d\xi} d\xi$$

Matematicamente, essa integral é a definição exata da **transposta da matriz que calculamos no Passo 1**! Ou seja, o termo de volume da nossa segunda equação é simplesmente $-\mathcal{S}_{sq1}^T$.

**Passo 4: O Termo de Fronteira (Boundterm) e a Matriz $\mathcal{S}_{sq2}$**  
O termo que restou da integração por partes é avaliado estritamente nas fronteiras da célula:

$$\text{Boundterm} = \left[ \sqrt{a(x)} \phi_i \right]_{x_L}^{x_R} = \sqrt{a(x_R)}\phi_i(x_R) - \sqrt{a(x_L)}\phi_i(x_L)$$

As avaliações podem ser feitas usando as **matrizes Frk** e **Flk**. Unindo essas avaliações com os coeficientes de viscosidade da borda do elemento (epsb), construímos o operador de fronteira:  

```Python
# Usando definicoes da Lift Matrices
Boundterm = np.sqrt(epsb[-1])*Frk - np.sqrt(epsb[0])*Flk
```

Por fim, o operador total que multiplicará o vetor auxiliar $q$ do lado direito da nossa forma fraca é a junção do termo de fronteira com a integral de volume negativa. Essa junção é a nossa segunda matriz modificada:

$$\mathcal{S}_{sq2} = \text{Boundterm} - \mathcal{S}_{sq1}^T$$

> No Código: Não precisamos recalcular uma nova matriz integral do zero, bastando agrupar os operadores com a linha:

```Ssq2 = Boundterm - Ssq1.T```


* Uma matriz de volume puramente interna ($\mathcal{S}_{sq1}$)
* Uma matriz que acopla a integração por partes com os termos de elevação da fronteira ($\mathcal{S}_{sq2}$)

---


#### **2. A Abordagem de Persson (Assimétrica e Rápida)**
Persson e Peraire adotam um caminho mais direto, sem dividir a viscosidade:
$$q = u_x \\
u_t + f(u)_x = (\epsilon q)_x$$

Neste caso, o cálculo de $q$ usa a nossa *Matriz de Rigidez Tradicional* intacta! A viscosidade $\epsilon$ só entra na segunda equação, exigindo a criação de apenas uma matriz modificada ($\mathcal{S}_{sq1}$) e barateando o custo computacional, embora perca a garantia formal de estabilidade incondicional.

> Abaixo, implementaremos a função `ModifStiffMatrix`, que engloba essas duas lógicas usando o parâmetro `type_form`.

In [ ]:
def ModifStiffMatrix(psi,Dpsi,wi,Nldof,Frk,Flk,epst, epsb, type_form='Warburton'):
    """
    Calcula a Matriz de Rigidez Modificada para o termo difusivo.

    Parâmetros:
      type_form: 'Warburton' (Simétrica, retorna Ssq1 e Ssq2)
                 'Persson'   (Assimétrica, retorna Ssq1 e None)
    """
    Ssq1 = np.zeros((Nldof, Nldof))

    if type_form == 'Warburton':
        # Abordagem Simétrica (Hesthaven-Warburton)
        Ssq1 = np.dot(np.sqrt(epst)*wi*psi.T, Dpsi)

        # Termo de fronteira para Ssq2
        Boundterm = (np.sqrt(epsb[-1])*Frk - np.sqrt(epsb[0])*Flk)
        Ssq2 = Boundterm - Ssq1.T

        return Ssq1.T, Ssq2.T

    elif type_form == 'Persson':
        # Abordagem Assimétrica (Persson-Peraire)
        Ssq1 = np.dot(epst*wi*psi.T, Dpsi)

        # Persson não precisa de uma segunda matriz modificada
        return Ssq1.T, None

    else:
        raise ValueError("Modelo de matriz modificada não reconhecido! Escolha 'Warburton' ou 'Persson'.")